# pypgo Mesh API Demo

This notebook is a compact tour of the current M1 mesh API:

- `pypgo.mesh`: `TriMeshData`, `TetMeshData`, `CubicMeshData`, OBJ I/O, and shape factories
- `pypgo.mesh.geo`: geometry facades, normals, and barycentric embedding
- `pypgo.mesh.veg`: Vega volume materials, `.veg` I/O, `VolumeMesh`, and surface extraction
- `pypgo.tools.mesh`: quality checks and mesher wrappers
- `pypgo.sparse` / `pypgo.sim`: sparse COO export and solver-ready mesh factories


In [16]:
import os
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pypgo as pgo

from pypgo.mesh import CubicMeshData, MeshDataType, TetMeshData, TriMeshData
from pypgo.mesh.geo import BarycentricEmbedding, CubicMeshGeo, TetMeshGeo, TriMeshGeo
from pypgo.mesh.veg import (
    ENuMaterial,
    MeshRegion,
    MeshSet,
    MooneyRivlinMaterial,
    VegFile,
    VolumeMesh,
)
from pypgo.tools.mesh import check_surface_quality, cubic_mesher, has_tetwild, tet_mesher


## Notebook visualization helpers

The demo uses real OBJ assets from `pypgo/examples/assets/obj`. PyVista is optional and only needed for interactive 3D views; install it with `pip install -e .[examples]`. The heavier `dragon.obj` asset is included for experimentation, but the default cells use smaller assets so the notebook stays quick to run.


In [17]:
def _find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "setup.py").exists() and (candidate / "pypgo").exists():
            return candidate
    raise RuntimeError("Could not find repository root from the current working directory")


REPO_ROOT = _find_repo_root()
ASSET_DIR = REPO_ROOT / "pypgo" / "examples" / "assets" / "obj"
PYVISTA_INSTALL_HINT = "Install visualization dependencies with: pip install -e .[examples]"

try:
    import pyvista as pv
except ModuleNotFoundError:
    pv = None
    print(f"PyVista is not installed. {PYVISTA_INSTALL_HINT}")


def _require_pyvista() -> bool:
    if pv is None:
        print(f"Skipping PyVista view. {PYVISTA_INSTALL_HINT}")
        return False
    return True


def to_pyvista_surface(surface_data: TriMeshData):
    if not isinstance(surface_data, TriMeshData):
        raise TypeError(f"surface_data must be a TriMeshData, got {type(surface_data).__name__}")
    if pv is None:
        raise RuntimeError(PYVISTA_INSTALL_HINT)

    faces = np.column_stack(
        [
            np.full(surface_data.num_elements, 3, dtype=np.int64),
            surface_data.elements,
        ]
    ).ravel()
    return pv.PolyData(surface_data.vertices, faces)


def to_pyvista_volume(volume_data):
    if pv is None:
        raise RuntimeError(PYVISTA_INSTALL_HINT)
    if not isinstance(volume_data, (TetMeshData, CubicMeshData)):
        raise TypeError(
            f"volume_data must be a TetMeshData or CubicMeshData, got {type(volume_data).__name__}"
        )

    elements = volume_data.elements
    width = elements.shape[1]
    if width == 4:
        cell_type = pv.CellType.TETRA
    elif width == 8:
        cell_type = pv.CellType.HEXAHEDRON
    else:
        raise ValueError(f"Unsupported volume element width: {width}")

    cells = np.column_stack(
        [
            np.full(volume_data.num_elements, width, dtype=np.int64),
            elements,
        ]
    ).ravel()
    cell_types = np.full(volume_data.num_elements, cell_type, dtype=np.uint8)
    return pv.UnstructuredGrid(cells, cell_types, volume_data.vertices)


def plot_surface(meshes, *, titles=None, show_edges=True, colors=None, window_size=(900, 360)):
    if not _require_pyvista():
        return None

    if isinstance(meshes, TriMeshData):
        meshes = [meshes]
    meshes = list(meshes)
    titles = titles or [None] * len(meshes)
    colors = colors or ["lightgray"] * len(meshes)

    plotter = pv.Plotter(shape=(1, len(meshes)), window_size=window_size)
    for index, mesh in enumerate(meshes):
        if len(meshes) > 1:
            plotter.subplot(0, index)
        plotter.add_mesh(
            to_pyvista_surface(mesh),
            color=colors[index % len(colors)],
            show_edges=show_edges,
            smooth_shading=False,
        )
        if titles[index]:
            plotter.add_text(titles[index], position="upper_left", font_size=10)
        plotter.view_isometric()
        plotter.camera.zoom(1.2)
    return plotter.show()


def plot_volume_surface(meshes, *, titles=None, show_edges=True, colors=None, window_size=(900, 360)):
    if not _require_pyvista():
        return None

    if isinstance(meshes, (TetMeshData, CubicMeshData)):
        meshes = [meshes]
    meshes = list(meshes)
    titles = titles or [None] * len(meshes)
    colors = colors or ["lightsteelblue"] * len(meshes)

    plotter = pv.Plotter(shape=(1, len(meshes)), window_size=window_size)
    for index, mesh in enumerate(meshes):
        if len(meshes) > 1:
            plotter.subplot(0, index)
        plotter.add_mesh(
            to_pyvista_volume(mesh).extract_surface(),
            color=colors[index % len(colors)],
            show_edges=show_edges,
            smooth_shading=False,
        )
        if titles[index]:
            plotter.add_text(titles[index], position="upper_left", font_size=10)
        plotter.view_isometric()
        plotter.camera.zoom(1.2)
    return plotter.show()


print("asset directory:", ASSET_DIR)
print("available OBJ assets:", sorted(path.name for path in ASSET_DIR.glob("*.obj")))


asset directory: /Users/jinceyang/Desktop/codebase/libpgo/pypgo/examples/assets/obj
available OBJ assets: ['box-with-sphere.obj', 'box.obj', 'bunny.obj', 'dragon.obj']


## 1. MeshData containers

`MeshData` is the canonical data representation at public API boundaries. Constructors accept list-like or NumPy input and normalize vertices to `float64` and elements to `int64`.


In [18]:
tri_data = TriMeshData(
    np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0]], dtype=np.float32),
    np.array([[0, 1, 2], [1, 3, 2]], dtype=np.int32),
)

tet_data = TetMeshData(
    np.array(
        [
            [0, 0, 0],
            [1, 0, 0],
            [0, 1, 0],
            [0, 0, 1],
        ],
        dtype=np.float64,
    ),
    np.array([[0, 1, 2, 3]], dtype=np.int64),
)

cubic_data = CubicMeshData(
    np.array(
        [
            [0, 0, 0],
            [1, 0, 0],
            [1, 1, 0],
            [0, 1, 0],
            [0, 0, 1],
            [1, 0, 1],
            [1, 1, 1],
            [0, 1, 1],
        ],
        dtype=np.float64,
    ),
    np.array([[0, 1, 2, 3, 4, 5, 6, 7]], dtype=np.int64),
)

print("tri:", tri_data.num_vertices, tri_data.num_elements, tri_data.vertices.dtype, tri_data.elements.dtype)
print("tet:", tet_data.num_vertices, tet_data.num_elements, tet_data.mesh_type == MeshDataType.Tet)
print("cubic:", cubic_data.num_vertices, cubic_data.num_elements, cubic_data.mesh_type == MeshDataType.Cubic)

plot_surface(tri_data, titles=["TriMeshData"])
plot_volume_surface([tet_data, cubic_data], titles=["TetMeshData", "CubicMeshData"])


tri: 4 2 float64 int64
tet: 4 1 True
cubic: 8 1 True


Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x3153e9a90_9&reconnect=auto" class="pyvista…

/var/folders/lw/j9_lmxtd6gx45dln0mlczmn80000gn/T/ipykernel_7065/3546682621.py:111: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  to_pyvista_volume(mesh).extract_surface(),


Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x31cd807d0_10&reconnect=auto" class="pyvist…

## 2. NumPy properties and shape factories

Derived properties are intentionally NumPy-friendly: bounding boxes, volumes, centers of mass, subsetting, concatenation, and triangle normals all come back as arrays or scalars. The surface examples below load real OBJ assets from `pypgo/examples/assets/obj`.


In [19]:
box = pgo.mesh.read_obj(str(ASSET_DIR / "box.obj"))
bunny = pgo.mesh.read_obj(str(ASSET_DIR / "bunny.obj"))
sphere = pgo.mesh.create_sphere(radius=1.0, axis_subdiv=12, height_subdiv=6)

bmin, bmax = box.bbox
box_geo = TriMeshGeo.from_mesh_data(box)

print("box.obj vertices/elements:", box.num_vertices, box.num_elements)
print("box bbox:", bmin, bmax)
print("bunny.obj vertices/elements:", bunny.num_vertices, bunny.num_elements)
print("sphere vertices/elements:", sphere.num_vertices, sphere.num_elements)
print("tet volume:", tet_data.volume)
print("cubic volume:", cubic_data.volume)
print("first 3 face normals:\n", box_geo.face_normals[:3])

plot_surface([box, bunny], titles=["box.obj", "bunny.obj"], show_edges=False)


box.obj vertices/elements: 194 384
box bbox: [-0.25      0.378135 -0.25    ] [0.25     0.878135 0.25    ]
bunny.obj vertices/elements: 4214 8424
sphere vertices/elements: 62 120
tet volume: 0.16666666666666666
cubic volume: 0.9999999999999999
first 3 face normals:
 [[ 0.  1.  0.]
 [ 1.  0.  0.]
 [ 0. -1.  0.]]


Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x127056c10_11&reconnect=auto" class="pyvist…

In [20]:
bunny_subset = bunny.take_elements(np.arange(0, bunny.num_elements, 4))
bunny_width = bunny.bbox[1][0] - bunny.bbox[0][0]
shifted_subset = TriMeshData(
    bunny_subset.vertices + np.array([1.4 * bunny_width, 0.0, 0.0]),
    bunny_subset.elements,
)
merged = TriMeshData.concatenate([bunny_subset, shifted_subset])

print("bunny subset elements:", bunny_subset.num_elements)
print("merged vertices/elements:", merged.num_vertices, merged.num_elements)
print("merged bbox:", merged.bbox)

plot_surface(
    [bunny, bunny_subset],
    titles=["original bunny.obj", "take_elements every 4th face"],
    show_edges=True,
    colors=["lightgray", "cornflowerblue"],
)
plot_surface(merged, titles=["concatenated shifted subsets"], show_edges=True, colors=["plum"])


bunny subset elements: 2106
merged vertices/elements: 8428 4212
merged bbox: (array([-0.094285,  0.033483, -0.060727]), array([0.2776334, 0.185332 , 0.058417 ]))


Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x311453610_12&reconnect=auto" class="pyvist…

Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x31cd81810_13&reconnect=auto" class="pyvist…

## 3. MeshGeo facades and explicit conversion

`MeshGeo` provides typed geometry accessors. Crossing between facade and data container is explicit: `.to_mesh_data()` and `.from_mesh_data(...)`.


In [21]:
tri_geo = TriMeshGeo.from_mesh_data(tri_data)
tet_geo = TetMeshGeo.from_mesh_data(tet_data)
cubic_geo = CubicMeshGeo.from_mesh_data(cubic_data)

print("triangles:\n", tri_geo.triangles)
print("tets:\n", tet_geo.tets)
print("cubes:\n", cubic_geo.cubes)
print("roundtrip:", np.array_equal(tri_geo.to_mesh_data().elements, tri_data.elements))


triangles:
 [[0 1 2]
 [1 3 2]]
tets:
 [[0 1 2 3]]
cubes:
 [[0 1 2 3 4 5 6 7]]
roundtrip: True


## 4. Surface quality checks

`check_surface_quality` combines pure NumPy checks with the `_core` self-intersection bridge.


In [37]:
box = pgo.mesh.create_box(bmin=[-0.1, -0.1, -0.1], bmax=[0.1, 0.1, 0.1])
bunny_with_bunny = TriMeshData.concatenate([bunny, box])
print("bunny:", bunny.num_vertices, bunny.num_elements)
print("box:", box.num_vertices, box.num_elements)
print("bunny bbox:", bunny.bbox)
print("box bbox:", box.bbox)
print("bunny_with_bunny bbox:", bunny_with_bunny.bbox)
print("bunny_with_bunny vertices/elements:", bunny_with_bunny.num_vertices, bunny_with_bunny.num_elements)
quality = check_surface_quality(bunny_with_bunny, short_edge_threshold=1e-6)
print(quality)
plot_surface(bunny, titles=["Bunny"], show_edges=True, colors=["lightcoral"])
plot_surface(box, titles=["Box"], show_edges=True, colors=["lightcoral"])
plot_surface(bunny_with_bunny, titles=["Bunny with Box"], show_edges=True, colors=["lightcoral"])


bunny: 4214 8424
box: 8 12
bunny bbox: (array([-0.094285,  0.033483, -0.060727]), array([0.060681, 0.185332, 0.058417]))
box bbox: (array([-0.1, -0.1, -0.1]), array([0.1, 0.1, 0.1]))
bunny_with_bunny bbox: (array([-0.1, -0.1, -0.1]), array([0.1     , 0.185332, 0.1     ]))
bunny_with_bunny vertices/elements: 4222 8436
QualityReport(is_clean=False, degenerate_tris=[], short_edges=[], non_manifold_edges=[], flipped_tris=[], has_self_intersections=True)


Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x32b7974d0_39&reconnect=auto" class="pyvist…

Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x32b797d90_40&reconnect=auto" class="pyvist…

Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x32bbcc690_41&reconnect=auto" class="pyvist…

## 5. Vega volume meshes and `.veg` I/O

`.veg` I/O preserves mesh data, material payloads, sets, and regions. The public object is `VegFile`; `VolumeMesh` is constructed from `TetMeshData` or `CubicMeshData`.


In [40]:
soft = ENuMaterial("soft", density=1000.0, E=2e6, nu=0.35)
insert = MooneyRivlinMaterial("insert", density=1200.0, mu01=3.0, mu10=4.0, v1=0.2)

two_tet_data = TetMeshData(
    np.array(
        [
            [0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 1.0],
            [1.0, 1.0, 1.0],
        ],
        dtype=np.float64,
    ),
    np.array([[0, 1, 2, 3], [1, 2, 3, 4]], dtype=np.int64),
)

veg = VegFile(
    mesh_data=two_tet_data,
    materials=[soft
               , insert],
    sets=[
        MeshSet("allElements", [0, 1]),
        MeshSet("softSet", [0]),
        MeshSet("insertSet", [1]),
    ],
    regions=[
        MeshRegion(0, 1),
        MeshRegion(1, 2),
    ],
)

volume = VolumeMesh(veg.mesh_data, regions=veg.to_volume_regions())
print("volume:", volume)
print("surface:", volume.extract_surface_mesh().num_elements, "triangles")
print("regions:", veg.to_volume_regions())

pgo.mesh.veg.write_veg("two_tet.veg", veg)
plot_surface(volume.extract_surface_mesh(), titles=["Volume Surface"], show_edges=True, colors=["lightseagreen"])


volume: VolumeMesh(type=MeshType.Tet, vertices=5, elements=2)
surface: 6 triangles
regions: [('softSet', ENuMaterial(name='soft', density=1000.0, E=2000000.0, nu=0.35), [0]), ('insertSet', MooneyRivlinMaterial(name='insert', density=1200.0, mu01=3.0, mu10=4.0, v1=0.2), [1])]


Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x3114502d0_42&reconnect=auto" class="pyvist…

In [24]:
tmpdir = tempfile.mkdtemp()
try:
    veg_path = os.path.join(tmpdir, "two_materials.veg")
    pgo.mesh.veg.write_veg(veg_path, veg)
    loaded = pgo.mesh.veg.read_veg(veg_path)

    print("loaded mesh:", type(loaded.mesh_data).__name__, loaded.mesh_data.num_elements)
    print("materials:", [type(m).__name__ + ':' + m.name for m in loaded.materials])
    print("sets:", [(s.name, s.elements) for s in loaded.sets])
    print("regions:", [(r.material_index, r.set_index) for r in loaded.regions])
finally:
    shutil.rmtree(tmpdir)


loaded mesh: TetMeshData 2
materials: ['ENuMaterial:soft', 'MooneyRivlinMaterial:insert']
sets: [('allElements', [0, 1]), ('softSet', [0]), ('insertSet', [1])]
regions: [(0, 1), (1, 2)]


## 6. Barycentric embedding and sparse COO

`BarycentricEmbedding` maps volume displacements to target locations. Its interpolation matrix is exposed through `pypgo.sparse.SparseMatrix.to_coo()`.


In [25]:
single_volume = VolumeMesh(tet_data, soft)
embedding = BarycentricEmbedding(np.array([[0.25, 0.25, 0.25]], dtype=np.float64), single_volume)
matrix = embedding.interpolation_matrix
rows, cols, values = matrix.to_coo()

displacement = np.zeros((single_volume.num_vertices, 3), dtype=np.float64)
displacement[:, 2] = np.linspace(0.0, 1.0, single_volume.num_vertices)

print("matrix shape/nnz:", matrix.shape, matrix.nnz)
print("coo:", rows, cols, values)
print("embedded displacement:", embedding.deform(displacement.ravel()))


matrix shape/nnz:12,3,12
 (3, 12) 12
coo: [0 0 0 0 1 1 1 1 2 2 2 2] [ 0  3  6  9  1  4  7 10  2  5  8 11] [0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25 0.25]
embedded displacement: [0.  0.  0.5]


## 7. Solver-ready meshes

Use explicit factories for solver-facing meshes. Volume and shell paths are separate on purpose.


In [26]:
sim_volume = pgo.sim.SimulationMesh.create_volumetric(single_volume)
shell_material = pgo.sim.KoiterStVKShellMaterial(
    "cloth", thickness=0.01, E_membrane=1e6, nu_membrane=0.35
)
sim_shell = pgo.sim.SimulationMesh.create_shell(tri_data, shell_material)

print("volume sim:", sim_volume.mesh_type, sim_volume.num_vertices, sim_volume.num_elements)
print("shell sim:", sim_shell.mesh_type, sim_shell.num_vertices, sim_shell.num_elements)


volume sim: tet 4 1
shell sim: shell 4 2


## 8. Mesher wrappers

The wrappers accept `TriMeshData` and return volume `MeshData`. `tetwild` is optional; `has_tetwild()` reflects the build configuration. This section uses the real `box-with-sphere.obj` asset as input.


In [ ]:
mesher_surface = pgo.mesh.read_obj(str(ASSET_DIR / "box-with-sphere.obj"))
cubic_from_surface = cubic_mesher(mesher_surface, resolution=20)
tet_from_surface = tet_mesher(mesher_surface, backend="tetgen", config={"command": "pq1.414a0.1"})

cubic_surface = VolumeMesh(cubic_from_surface, soft).extract_surface_mesh()
tet_surface = VolumeMesh(tet_from_surface, soft).extract_surface_mesh()

print("mesher input surface:", mesher_surface.num_vertices, mesher_surface.num_elements)
print("cubic mesher:", cubic_from_surface.num_vertices, cubic_from_surface.num_elements)
print("tet mesher:", tet_from_surface.num_vertices, tet_from_surface.num_elements)
print("cubic extracted surface:", cubic_surface.num_vertices, cubic_surface.num_elements)
print("tet extracted surface:", tet_surface.num_vertices, tet_surface.num_elements)
print("tetwild enabled:", has_tetwild())

plot_surface(mesher_surface, titles=["box-with-sphere.obj input"], show_edges=False)
plot_volume_surface(
    [cubic_from_surface, tet_from_surface],
    titles=["cubic_mesher volume surface", "tet_mesher volume surface"],
    show_edges=True,
    colors=["lightsteelblue", "palegreen"],
)
plot_surface(
    [cubic_surface, tet_surface],
    titles=["VolumeMesh(cubic).extract_surface_mesh()", "VolumeMesh(tet).extract_surface_mesh()"],
    show_edges=True,
    colors=["lightskyblue", "mediumseagreen"],
)


Saved mesh (#v: 3245, #t: 6482) to /var/folders/lw/j9_lmxtd6gx45dln0mlczmn80000gn/T/libpgo-pypgo-mesher-174662248209125.obj.
Saved mesh (#v: 3245, #t: 6482) to /var/folders/lw/j9_lmxtd6gx45dln0mlczmn80000gn/T/libpgo-pypgo-mesher-174662344032041.obj.
Tetrahedralizing surface mesh...
Delaunizing vertices...
Delaunay seconds:  0.007387
Creating surface mesh ...
Surface mesh seconds:  0.002513
Recovering boundaries...
Boundary recovery seconds:  0.005137
Removing exterior tetrahedra ...
Exterior tets removal seconds:  0.000971
Recovering Delaunayness...
Delaunay recovery seconds:  0.001524
Refining mesh...
Refinement seconds:  0.021389
Smoothing vertices...
Mesh smoothing seconds:  0.025011
Improving mesh...
Mesh improvement seconds:  0.001746

Writing nodes.
Writing elements.
Writing faces.
Writing edges.

Output seconds:  0.000473
Total running seconds:  0.066265

Statistics:

  Input points: 3245
  Input facets: 6482
  Input segments: 1131
  Input holes: 0
  Input regions: 0

  Mesh poi

Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x32b794690_43&reconnect=auto" class="pyvist…

/var/folders/lw/j9_lmxtd6gx45dln0mlczmn80000gn/T/ipykernel_7065/3546682621.py:111: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  to_pyvista_volume(mesh).extract_surface(),


Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x32bbce490_44&reconnect=auto" class="pyvist…

Widget(value='<iframe src="http://localhost:52056/index.html?ui=P_0x32bbcec10_45&reconnect=auto" class="pyvist…